# 4a · Chunk & embed the manuals

**Core · Live-code · ~20 min**

Turn the five Orbital manuals into a searchable vector store with Chroma.

> Needs Ollama with `nomic-embed-text`. Solution: [`solution/04a_embed_docs.ipynb`](solution/04a_embed_docs.ipynb).

In [ ]:
import sys, os, re
from pathlib import Path
sys.path.insert(0, os.path.abspath(".."))
from shared.llm import embed

manuals = sorted(Path("../data/manuals").glob("*.md"))
print([m.name for m in manuals])

## Chunking

LLMs work best with focused context. Instead of dumping a whole manual, we split each
document into **sections** (by `##` headings) so we can retrieve just the relevant part.

In [ ]:
def chunk_markdown(text, source):
    parts = re.split(r"\n(?=#{1,3}\s)", text)   # split before headings
    return [{"text": p.strip(), "source": source} for p in parts if len(p.strip()) >= 40]

# TODO: build a list `chunks` by chunking every manual
chunks = []
# for m in manuals:
#     chunks += chunk_markdown(m.read_text(), m.name)
print(f"{len(chunks)} chunks")

## Embed & store in Chroma

We compute embeddings ourselves (with Ollama) and hand them to Chroma, so nothing needs
to download.

In [ ]:
import chromadb

# TODO: embed every chunk's text -> `vectors`
# vectors = embed([c["text"] for c in chunks])

client = chromadb.EphemeralClient()
collection = client.create_collection("orbital_manuals")
# TODO: add ids, embeddings=vectors, documents, and metadatas (source) to the collection
# collection.add(ids=[f"c{i}" for i in range(len(chunks))], embeddings=vectors,
#                documents=[c["text"] for c in chunks],
#                metadatas=[{"source": c["source"]} for c in chunks])
print("stored:", collection.count())

In **4b** we'll query this collection to answer questions. (There's also a ready-made
`shared/rag.py` that wraps all of this — we'll use it in Modules 5 and 7.)